# Project 11: does dengue in Suriname rise after heavy rain?

**The question.** Do more dengue cases follow the weeks after heavy rain in Suriname? If the record shows a link, is it stronger than chance alone would produce?

**The audience.** A neighbourhood community group in Paramaribo that organises clean-ups of standing water. The group gets a one-page notice that says what the record shows about rain and dengue, how sure anyone can be, and what the World Health Organization advises households to do.

**The data.** Two public tables, joined by week. Weekly dengue cases for all of Suriname come from the OpenDengue database (version 1.3, built from PAHO's weekly reports). Daily rainfall for the model grid cell nearest Paramaribo comes from the Open-Meteo archive (the ERA5 reanalysis), summed into the same weeks. Both were downloaded on 2026-11-06.

**Your team.** Four roles rotate every week: builder, data checker, AI checker and presenter.

**What you hand in on Friday.** This notebook with every CHECK passing, one chart, the one-page notice for the community group, and your AI log.

## Load the data

This cell is given. It reads the table from the course folder, or builds it from the two public sources if the file is not there. Each row is one week, starting on a Sunday. `cases` is the number of dengue cases reported that week. `zero_filled` is True when the database filled in a zero because no report came in. `rain_mm` is the rain that fell that week, in millimetres.

In [ ]:
import io, json, zipfile, urllib.request
import numpy as np
import pandas as pd

DENGUE_URL = "https://opendengue.org/assets/Temporal_extract_PAHO_V1_3.zip"
RAIN_URL = ("https://archive-api.open-meteo.com/v1/archive?latitude=5.85&longitude=-55.2"
            "&start_date=2017-12-31&end_date=2024-12-28&daily=precipitation_sum"
            "&timezone=America%2FParamaribo&models=era5")
try:
    dengue = pd.read_csv("https://santerre.ai/bigi-data/data/week11-dengue-rain.csv", parse_dates=["week_start"])
except FileNotFoundError:
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(DENGUE_URL).read()))
    d = pd.read_csv(z.open(z.namelist()[0]), low_memory=False)
    d = d[(d["adm_0_name"] == "SURINAME") & (d["T_res"] == "Week")]
    d = pd.DataFrame({"week_start": pd.to_datetime(d["calendar_start_date"]),
                      "cases": d["dengue_total"].astype(int),
                      "zero_filled": d["UUID"].str.contains("Zero filling")})
    d = d[(d["week_start"] >= "2017-12-31") & (d["week_start"] <= "2024-12-22")]
    daily = json.load(urllib.request.urlopen(RAIN_URL))["daily"]
    rain = pd.DataFrame({"day": pd.to_datetime(daily["time"]), "mm": daily["precipitation_sum"]})
    rain["week_start"] = rain["day"] - pd.to_timedelta((rain["day"].dt.dayofweek + 1) % 7, unit="D")
    rain = rain.groupby("week_start")["mm"].sum().round(1).rename("rain_mm").reset_index()
    dengue = d.merge(rain, on="week_start", how="left").sort_values("week_start").reset_index(drop=True)
dengue

## TODO 1: describe the table

Print the number of rows, the first and last week, and the total of `cases`. Store in `n_zero_filled` the number of weeks the database filled with zero.

A filled-in zero is a week nobody reported. It is not a week with no dengue.

In [ ]:
# TODO 1
n_zero_filled = None

In [ ]:
# CHECK 1
assert n_zero_filled is not None, "n_zero_filled is still empty"
assert n_zero_filled == int(dengue["zero_filled"].sum())
assert (dengue["week_start"].diff().dropna() == pd.Timedelta(weeks=1)).all()
print("CHECK 1 passed:", n_zero_filled, "weeks were filled with zero")

## TODO 2: the rain before each week

Rain does not make a dengue case the same week. The US Centers for Disease Control and Prevention say a mosquito egg needs 7 to 10 days to become an adult, and the World Health Organization says symptoms begin 4 to 10 days after infection. Reports take more time.

The project fixes its window before looking at the results: the total rain in weeks 3 to 6 before each week. Add that total as a column `rain_before`. Compute it on the full table, before you drop any week, so that every week's rain stays in its right place.

In [ ]:
# TODO 2

In [ ]:
# CHECK 2
assert "rain_before" in dengue, "add the column rain_before"
dates = dengue.set_index("week_start")["rain_mm"]
for i in [6, 120, 300]:
    t = dengue["week_start"][i]
    expected = sum(dates[t - pd.Timedelta(weeks=k)] for k in range(3, 7))
    assert abs(dengue["rain_before"][i] - expected) < 0.05, f"row {i} is off"
assert dengue["rain_before"].isna().sum() == 6
print("CHECK 2 passed: the week of", dengue["week_start"][120].date(), "had", round(dengue["rain_before"][120], 1), "mm in weeks 3 to 6 before it")

## TODO 3: keep the weeks you can use

Store in `weeks` the rows that were reported (not zero-filled) and have a value for `rain_before`.

In [ ]:
# TODO 3
weeks = None

In [ ]:
# CHECK 3
assert weeks is not None, "weeks is still empty"
assert not weeks["zero_filled"].any() and weeks["rain_before"].notna().all()
assert len(weeks) == int((~dengue["zero_filled"] & dengue["rain_before"].notna()).sum())
print("CHECK 3 passed:", len(weeks), "weeks to work with")

## TODO 4: chart cases and rain

Draw two charts, one above the other, with the same time axis: dengue cases per week and rain per week. Give each a title and labelled axes, and name both sources. Mark where the zero-filled weeks are, or leave them out, and say which you did.

In [ ]:
# TODO 4

## TODO 5: the gap

Call a week **wet before** when its `rain_before` is above the median of `rain_before` in `weeks`. Store those labels as a True/False array in `wet`. Store in `gap` the mean cases in wet-before weeks minus the mean cases in the other weeks.

In [ ]:
# TODO 5
wet = None
gap = None

In [ ]:
# CHECK 5
cases = weeks["cases"].to_numpy()
w = (weeks["rain_before"] > weeks["rain_before"].median()).to_numpy()
assert wet is not None and gap is not None, "wet or gap is still empty"
assert (np.asarray(wet) == w).all(), "wet should be True above the median"
assert abs(gap - (cases[w].mean() - cases[~w].mean())) < 1e-9, "gap should be wet minus the other weeks"
print("CHECK 5 passed: gap of", round(gap, 2), "cases a week")

## TODO 6: the ordinary shuffle

Shuffle the `wet` labels 10,000 times with `np.random.default_rng(11)`, recompute the gap each time, and store in `p_shuffle` the share of shuffles whose gap is at least as large as `gap` in either direction.

In [ ]:
# TODO 6
p_shuffle = None

In [ ]:
# CHECK 6
assert p_shuffle is not None, "p_shuffle is still empty"
checker = np.random.default_rng(5)
ref = np.array([(lambda s: cases[s].mean() - cases[~s].mean())(checker.permutation(w)) for _ in range(10000)])
p_ref = np.mean(np.abs(ref) >= abs(gap))
assert abs(p_shuffle - p_ref) < 0.005, "p_shuffle is off: count both directions"
print("CHECK 6 passed: p_shuffle close to", round(p_ref, 4))

## TODO 7: the shift test

Dengue cases drift from week to week and so does rain, so an ordinary shuffle breaks the order that the data carry. Keep `cases` in place and slide the `wet` labels past them with `np.roll`, for every shift from 26 weeks to `len(weeks) - 26` weeks. Store in `p_shift` the share of shifts whose gap is at least as large as `gap` in either direction.

In [ ]:
# TODO 7
p_shift = None

In [ ]:
# CHECK 7
assert p_shift is not None, "p_shift is still empty"
shifted = np.array([(lambda s: cases[s].mean() - cases[~s].mean())(np.roll(w, k)) for k in range(26, len(w) - 25)])
assert abs(p_shift - np.mean(np.abs(shifted) >= abs(gap))) < 1e-9, "p_shift is off: check the range of shifts"
print("CHECK 7 passed:", len(shifted), "shifts, p_shift", round(p_shift, 4))

## TODO 8: the AI check

Ask your AI assistant, without showing it the data: *"Do dengue cases in Suriname rise after heavy rain? How many weeks later?"* Write its answer and the delay it names in your AI log.

Then test it. Build a table `lag_table` with one row for each delay from 0 to 12 weeks and three columns: `lag`, `gap` (the wet-before gap when "wet" uses only the rain of the single week that many weeks earlier) and `p_shift` (the shift test for that gap). Use the same rows as `weeks`, dropping any whose earlier rain is missing. Compare the delay the AI named with your table.

Last, count your tries. You tested 13 delays, so the smallest p-value in the table is a winner picked from many. Say in the log what that means for any delay that looks strong.

In [ ]:
# TODO 8
lag_table = None

In [ ]:
# CHECK 8
assert lag_table is not None, "lag_table is still empty"
assert list(lag_table["lag"]) == list(range(13))
for k in [0, 4, 12]:
    lagged = dengue["rain_mm"].shift(k).loc[weeks.index]
    ok = lagged.notna().to_numpy()
    c, s = cases[ok], (lagged[ok] > lagged[ok].median()).to_numpy()
    assert abs(lag_table["gap"][k] - (c[s].mean() - c[~s].mean())) < 0.01, f"the gap at lag {k} is off"
print("CHECK 8 passed: lags with more cases after wet weeks:", list(lag_table.loc[lag_table["gap"] > 0, "lag"]))

## TODO 9: the notice for the community group

In a text cell below, write a notice of about 250 words for the neighbourhood community group. Say what the record shows about rain and dengue, and say how often chance alone would produce it in words a resident would use. Name both sources. Say two things the data cannot tell the group, and repeat the World Health Organization's advice on water storage containers with its link.